# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


# Na 3.0

## Generacja 10 losowych przypadków
### Parametry:
* 3 plecaki
* 10 zestawów parametrów
* największy - conajmniej 10 sek.
* optymalne rozwiązanie: 
    * kady plecak ma pojemośc 2 razy lub więcej większą od wagi kazdego przedmiotu
    * suma pojemnosci plecakow < suma wag - 2 najwieksze wagi

In [1]:
using Distributions

using Statistics

const GLOBAL_SEED = 20260414

function set_seed!(seed::Int=GLOBAL_SEED)
    Random.seed!(seed)
    return seed
end

function make_dzn(n::Int, n_knapsacks::Int; seed::Int=GLOBAL_SEED)
    rng = MersenneTwister(seed)
    profits = rand(rng, 10:1000, n)
    weights = rand(rng, 10:100, n)
    capacities = rand(rng, 100:300, n_knapsacks)
    content = """
    ITEM = _(1..$n);
    knapsack_n = $n_knapsacks;
    capacities = $capacities;
    profits = $profits;
    weights = $weights;
    """
    open("knapsack_generated.dzn", "w") do io
        write(io, content)
    end
end

make_dzn (generic function with 1 method)

In [2]:
struct KnapsackProblem
    capacities::Vector{Int}
    weights::Vector{Int}
    profits::Vector{Int}
end

function generate_problem(n_knapsacks::Int=3, n_items::Int=100; seed::Int=GLOBAL_SEED)
    rng = MersenneTwister(seed)
    capacities = rand(rng, 2500:3000, n_knapsacks)
    profits = rand(rng, 10:1000, n_items)
    weights = rand(rng, 10:100, n_items)
    return KnapsackProblem(capacities, weights, profits)
end

generate_problem (generic function with 3 methods)

Tworzenie zestawu danych

In [3]:
using Random
using Printf

# Pomocniczy parser: liczby całkowite z zapisu [1, 2, 3].
function _parse_int_vector(txt::AbstractString)
    m = collect(eachmatch(r"-?\d+", txt))
    return [parse(Int, mm.match) for mm in m]
end

# Wczytanie wartości skalarnej z pliku DZN, np. knapsack_n = 3;
function _parse_dzn_scalar_int(content::String, key::String)
    rx = Regex(key * raw"\s*=\s*(-?\d+)\s*;")
    m = match(rx, content)
    m === nothing && error("Brak pola $(key) w pliku DZN")
    return parse(Int, m.captures[1])
end

# Wczytanie tablicy z pliku DZN, np. capacities = [10, 20, 30];
function _parse_dzn_vector_int(content::String, key::String)
    rx = Regex(key * raw"\s*=\s*\[(.*?)\]\s*;", "s")
    m = match(rx, content)
    m === nothing && error("Brak tablicy $(key) w pliku DZN")
    return _parse_int_vector(m.captures[1])
end

# Wczytanie problemu plecakowego z pliku DZN
function load_knapsack_from_dzn(path::String)
    content = read(path, String)
    capacities = _parse_dzn_vector_int(content, "capacities")
    profits = _parse_dzn_vector_int(content, "profits")
    weights = _parse_dzn_vector_int(content, "weights")
    return KnapsackProblem(capacities, weights, profits)
end

function make_dzn_three_categories(path::String, n_items::Int, n_knapsacks::Int=3; rng=Random.default_rng())
    n_items >= 8 || error("Dla warunku \"co najmniej 2 poza plecakami\" potrzebujemy >= 8 przedmiotów")

    # Trzy kategorie: małe, średnie, duże.
    n_small = max(2, round(Int, 0.35 * n_items))
    n_medium = max(2, round(Int, 0.35 * n_items))
    n_large = n_items - n_small - n_medium
    n_large < 2 && (n_large = 2; n_medium = n_items - n_small - n_large)

    small_weights = rand(rng, 3:12, n_small)
    medium_weights = rand(rng, 13:35, n_medium)
    large_weights = rand(rng, 36:90, n_large)

    small_profits = rand(rng, 8:40, n_small)
    medium_profits = rand(rng, 45:170, n_medium)
    large_profits = rand(rng, 180:620, n_large)

    weights = vcat(small_weights, medium_weights, large_weights)
    profits = vcat(small_profits, medium_profits, large_profits)

    perm = randperm(rng, n_items)
    weights = weights[perm]
    profits = profits[perm]

    total_weight = sum(weights)
    max_weight = maximum(weights)

    # Pojemności dobieramy tak, aby:
    # 1) każdy plecak mógł pomieścić min. 2 przedmioty,
    # 2) łączna pojemność była wyraźnie mniejsza niż suma wag (zostają elementy poza plecakami).
    min_cap = 2 * max_weight + 5
    center = max(min_cap, round(Int, 0.56 * total_weight / n_knapsacks))
    capacities = [max(min_cap, center + rand(rng, -15:15)) for _ in 1:n_knapsacks]

    content = """
    ITEM = _(1..$n_items);
    knapsack_n = $n_knapsacks;
    capacities = $(repr(capacities));
    profits = $(repr(profits));
    weights = $(repr(weights));
    """

    open(path, "w") do io
        write(io, content)
    end

    return (capacities=capacities, weights=weights, profits=profits)
end

make_dzn_three_categories (generic function with 2 methods)

In [4]:
# Funkcja weryfikująca, czy przypisanie przedmiotów do plecaków spełnia warunki: 
# co najmniej 2 przedmioty w każdym plecaku i co najmniej 2 przedmioty poza plecakami.
function verify_assignment_rules(assignment::Vector{Int}, n_knapsacks::Int)
    counts = [count(==(k), assignment) for k in 1:n_knapsacks]
    outside = count(==(0), assignment)
    ok = all(c -> c >= 2, counts) && outside >= 2
    return (ok=ok, items_in_knapsack=counts, items_outside=outside)
end

function generate_report_problem_set(problem_specs::Vector{Tuple{String,Int}};
    problems_dir::String="problems",
    model_path::String="knapsack_report.mzn",
    max_attempts_per_case::Int=60,
    largest_min_seconds::Float64=10.0,
    solver::String="coinbc",
    seed::Int=GLOBAL_SEED)

    mkpath(problems_dir)
    accepted = String[]
    base_rng = MersenneTwister(seed)

    for (idx, (name, n_items)) in enumerate(problem_specs)
        target_seconds = idx == length(problem_specs) ? largest_min_seconds : 0.0
        accepted_case = false
        local current_items = n_items

        for attempt in 1:max_attempts_per_case
            dzn_path = joinpath(problems_dir, "$(name).dzn")
            make_dzn_three_categories(dzn_path, current_items, 3; rng=base_rng)

            mz = run_minizinc_from_julia(model_path, dzn_path; solver=solver)
            if mz.profit === nothing || isempty(mz.assignment)
                if idx == length(problem_specs) && attempt % 10 == 0
                    current_items = round(Int, 1.15 * current_items)
                end
                continue
            end

            local_check = verify_assignment_rules(mz.assignment, 3)
            ok = mz.packing_rules_ok && local_check.ok && (mz.elapsed_seconds >= target_seconds)

            if ok
                push!(accepted, dzn_path)
                accepted_case = true
                break
            end

            if idx == length(problem_specs) && attempt % 10 == 0
                current_items = round(Int, 1.15 * current_items)
            end
        end

        if !accepted_case
            println("[WARN] Nie udało się zaakceptować przypadku $(name)") # jeżeli rowiązanie nie spełnia warunków - wyskakuje warning
        end
    end

    return accepted
end

generate_report_problem_set (generic function with 1 method)

In [5]:
function run_minizinc_from_julia(model_path::String, dzn_path::String; solver::String="coinbc", time_limit_ms::Int=0)
    cmd_with_solver = time_limit_ms > 0 ?
        `minizinc --solver $solver --time-limit $time_limit_ms $model_path $dzn_path` :
        `minizinc --solver $solver $model_path $dzn_path`
    cmd_default = time_limit_ms > 0 ?
        `minizinc --time-limit $time_limit_ms $model_path $dzn_path` :
        `minizinc $model_path $dzn_path`

    t0 = time()
    raw = ""
    try
        raw = read(cmd_with_solver, String)
    catch e
        msg = sprint(showerror, e)
        if occursin("no solver with tag", lowercase(msg))
            raw = read(cmd_default, String)
        else
            rethrow(e)
        end
    end
    elapsed = time() - t0

    function get_line_value(key::String)
        rx = Regex("^" * key * "=(.*)\$", "m")
        m = match(rx, raw)
        m === nothing && return nothing
        return strip(m.captures[1])
    end

    profit_txt = get_line_value("profit")
    x_txt = get_line_value("x")
    in_txt = get_line_value("items_in_knapsack")
    out_txt = get_line_value("items_outside")
    ok_txt = get_line_value("packing_rules_ok")

    return (
        raw_output = raw,
        elapsed_seconds = elapsed,
        profit = profit_txt === nothing ? nothing : parse(Int, profit_txt),
        assignment = x_txt === nothing ? Int[] : _parse_int_vector(x_txt),
        items_in_knapsack = in_txt === nothing ? Int[] : _parse_int_vector(in_txt),
        items_outside = out_txt === nothing ? nothing : parse(Int, out_txt),
        packing_rules_ok = ok_txt === nothing ? false : lowercase(ok_txt) == "true"
    )
end

run_minizinc_from_julia (generic function with 1 method)

In [7]:
problem_specs = [
    ("kp_01", 35),
    ("kp_02", 45),
    ("kp_03", 55),
    ("kp_04", 70),
    ("kp_05", 85),
    ("kp_06", 100),
    ("kp_07", 120),
    ("kp_08", 145),
    ("kp_09", 170),
    ("kp_10", 210)
]

generated_files = generate_report_problem_set(problem_specs;
    problems_dir="problems",
    model_path="knapsack_report.mzn",
    max_attempts_per_case=80,
    largest_min_seconds=10.0,
    solver="coinbc"
)

println("\nZaakceptowane problemy:")
foreach(println, generated_files)


Zaakceptowane problemy:
problems\kp_01.dzn
problems\kp_02.dzn
problems\kp_03.dzn
problems\kp_04.dzn
problems\kp_05.dzn
problems\kp_06.dzn
problems\kp_07.dzn
problems\kp_08.dzn
problems\kp_09.dzn
problems\kp_10.dzn


In [6]:
generated_files = [
    "problems/kp_01.dzn",
    "problems/kp_02.dzn",
    "problems/kp_03.dzn",
    "problems/kp_04.dzn",
    "problems/kp_05.dzn",
    "problems/kp_06.dzn",
    "problems/kp_07.dzn",
    "problems/kp_08.dzn",
    "problems/kp_09.dzn",
    "problems/kp_10.dzn"
]

10-element Vector{String}:
 "problems/kp_01.dzn"
 "problems/kp_02.dzn"
 "problems/kp_03.dzn"
 "problems/kp_04.dzn"
 "problems/kp_05.dzn"
 "problems/kp_06.dzn"
 "problems/kp_07.dzn"
 "problems/kp_08.dzn"
 "problems/kp_09.dzn"
 "problems/kp_10.dzn"

## TabuSearch

In [ ]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    iter_best_move = 0
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                continue
            end
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        if best_move == 0
            break
        end
        chosen_move = moves[best_move]
        item_id = chosen_move[1]
        prev_bin = s.current[item_id]
        apply!(s.current, chosen_move)
        push!(s.tabu_buffer, invert_move(p, moves[best_move], prev_bin))
        if best_move_obj < s.best_seen_obj
            iter_best_move = s.iter
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen, iter_best_move
end


function objective(p::KnapsackProblem, x::Vector{Int})
    profit = 0
    for i in eachindex(x)
        if x[i] > 0
            profit += p.profits[i]
        end
    end
    return -profit
end


function apply!(x, move::Tuple{Int,Int})
    item_idx, new_bin = move
    x[item_idx] = new_bin
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Int,Int}, old_bin::Int)
    return (move[1], old_bin)
end


function possible_moves(p::KnapsackProblem, x::Vector{Int})
    move_list = Tuple{Int,Int}[]
    current_weights = zeros(Int, length(p.capacities))
    for (item_id, bin_id) in enumerate(x)
        if bin_id > 0 # 0 oznacza brak przypisania
            current_weights[bin_id] += p.weights[item_id]
        end
    end    

    for i in eachindex(x)
        if x[i] > 0
            # możliwość usunięcia i z plecaka x[i]
            push!(move_list, (i, 0))
        end
        for k in eachindex(p.capacities)
            # przeniesienie do plecaka
            if x[i] == 0 && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i,k))
            end
        end
    end

    return move_list
end


possible_moves (generic function with 1 method)

In [ ]:
function test(kp)
    x0 = fill(0, length(kp.weights))
    st = TabuState(kp, x0; buffer_length=10)
    sol, iter_num = solve_tabu(kp, st; iteration_limit=1000000)

    println("Pełne rozwiązanie: ", sol)
    println("Iteracja w której zostało osiągnięte najlepsze rozwiązanie: ", iter_num)
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end
@time begin
test(kp1)
end



Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 196
Pełne rozwiązanie: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 3, 1, 0, 2, 0, 0, 0, 0, 3, 1, 1, 3, 0, 2, 0, 1, 0, 2, 0, 0, 0, 0, 0, 3, 2, 1, 0, 0, 1, 0]
Best objective: -1882
Last iteration: 1000000
  2.625209 seconds (6.22 M allocations: 1018.126 MiB, 3.13% gc time, 2.49% compilation time)


### 3.3 - rózne długości linii zakazów

In [ ]:
function experiment(kp)
    dlugosci = [1, 2, 5]
    
    for L in dlugosci
        @time begin
        println("\n--- Eksperyment dla długości tabu L = $L ---")
        
        x0 = fill(0, length(kp.weights))
        st = TabuState(kp, x0; buffer_length = L)
        
        sol, iter_num = solve_tabu(kp, st; iteration_limit = 100000)
        
        println("Pełne rozwiązanie: ", sol)
        println("Iteracja w której zostało osiągnięte najlepsze rozwiązanie: ", iter_num)

        println("Best objective: ", -st.best_seen_obj)
        println("Last iteration: ", st.iter)
        end
    end
    return -st.best_seen_obj
end
for file in generated_files
    println("\n=== Rozwiązanie problemu z pliku: $(basename(file)) ===")
    kp = load_knapsack_from_dzn(file)
end


=== Rozwiązanie problemu z pliku: kp_01.dzn ===

--- Eksperyment dla długości tabu L = 1 ---
Pełne rozwiązanie: [0, 3, 0, 0, 0, 3, 3, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 2, 1, 0, 0, 1, 0, 1, 0, 0, 0, 3, 1, 0, 2, 2]
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 14
Best objective: -4017
Last iteration: 100000
  0.323974 seconds (1.25 M allocations: 123.572 MiB, 7.94% gc time, 75.50% compilation time)

--- Eksperyment dla długości tabu L = 2 ---
Pełne rozwiązanie: [0, 3, 0, 0, 0, 3, 3, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 2, 1, 0, 0, 1, 0, 1, 0, 0, 0, 3, 1, 0, 2, 2]
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 14
Best objective: -4017
Last iteration: 100000
  0.105743 seconds (600.43 k allocations: 90.068 MiB, 13.99% gc time)

--- Eksperyment dla długości tabu L = 5 ---
Pełne rozwiązanie: [0, 3, 0, 0, 0, 3, 3, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 2, 1, 0, 0, 1, 0, 1, 0, 0, 0, 3, 1, 0, 2, 2]
Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 14

In [17]:
using Statistics

function run_full_comparison(files)
    dlugosci = [1, 2, 5]
    
    println("\n| Plik | L | Najlepszy Cel | Iteracja (Best) | Czas [s] |")
    println("|------|---|---------------|-----------------|----------|")

    for file in files
        kp = load_knapsack_from_dzn(file)
        filename = basename(file)

        for L in dlugosci
            stats = @timed begin
                x0 = fill(0, length(kp.weights))
                st = TabuState(kp, x0; buffer_length = L)
                sol, iter_num = solve_tabu(kp, st; iteration_limit = 100000)
                (obj = -st.best_seen_obj, best_iter = iter_num)
            end

            @printf("| %s | %d | %.2f | %d | %.4f |\n", 
                    filename, L, stats.value.obj, stats.value.best_iter, stats.time)
        end
    end
end

run_full_comparison(generated_files)


| Plik | L | Najlepszy Cel | Iteracja (Best) | Czas [s] |
|------|---|---------------|-----------------|----------|
| kp_01.dzn | 1 | 4017.00 | 14 | 0.0722 |
| kp_01.dzn | 2 | 4017.00 | 14 | 0.0585 |
| kp_01.dzn | 5 | 4017.00 | 14 | 0.0711 |
| kp_02.dzn | 1 | 4560.00 | 25 | 0.0650 |
| kp_02.dzn | 2 | 4545.00 | 16 | 0.0735 |
| kp_02.dzn | 5 | 4555.00 | 23 | 0.0766 |
| kp_03.dzn | 1 | 6707.00 | 21 | 0.0946 |
| kp_03.dzn | 2 | 6714.00 | 29 | 0.0931 |
| kp_03.dzn | 5 | 6744.00 | 47 | 0.1234 |
| kp_04.dzn | 1 | 8832.00 | 29 | 0.1205 |
| kp_04.dzn | 2 | 8863.00 | 32 | 0.1147 |
| kp_04.dzn | 5 | 9010.00 | 26 | 0.1696 |
| kp_05.dzn | 1 | 10401.00 | 26 | 0.1780 |
| kp_05.dzn | 2 | 10401.00 | 26 | 0.1671 |
| kp_05.dzn | 5 | 10401.00 | 26 | 0.2202 |
| kp_06.dzn | 1 | 11818.00 | 32 | 0.2464 |
| kp_06.dzn | 2 | 11818.00 | 32 | 0.2367 |
| kp_06.dzn | 5 | 11839.00 | 44 | 0.2191 |
| kp_07.dzn | 1 | 13959.00 | 37 | 0.2984 |
| kp_07.dzn | 2 | 13959.00 | 37 | 0.3045 |
| kp_07.dzn | 5 | 14090.00 | 40 | 0

Jak widać po załączonej tabeli zastosowanie dłuższej listy zakazów skutkowało lepszymi rozwiązaniami. Jedyny przypadek w którym lista zakazów równa 1 okazała się najlepsza to 2, gdzie osiągnęła wynik o 5 lepszy niż ta o długości 5. Nalezy też zauważyć, że liczba iteracji potrzebne do osiągnięcia najlepszego rozwiązania wydłuża się wraz z wydłużaniem się listy zakazów.

| Zestaw parametrów	| Tabu_L1 | Tabu_L2	| Tabu_L5 | 
| :--- | ------- | ------- | ------- |
| kp_01.dzn	| 4017.0 (0.072s) | 4017.0 (0.058s) | 4017.0 (0.071s) | 
| kp_02.dzn | 4560.0 (0.065s) | 4545.0 (0.073s) | 4555.0 (0.076s) | 
| kp_03.dzn	| 6707.0 (0.094s) | 6714.0 (0.093s) | 6744.0 (0.123s) | 
| kp_04.dzn	| 8832.0 (0.120s) | 8863.0 (0.114s) | 9010.0 (0.169s) | 
| kp_05.dzn	| 10401.0 (0.178s) | 10401.0 (0.167s) | 10401.0 (0.220s) | 
| kp_06.dzn	| 11818.0 (0.246s) | 11818.0 (0.236s) | 11839.0 (0.219s) | 
| kp_07.dzn	| 13959.0 (0.298s) | 13959.0 (0.304s) | 14090.0 (0.392s) | 
| kp_08.dzn	| 15809.0 (0.607s) | 15862.0 (0.491s) | 15954.0 (0.557s) | 
| kp_09.dzn	| 19692.0 (0.658s) | 19715.0 (0.527s) | 19787.0 (0.613s) | 
| kp_10.dzn	| 25100.0 (0.879s) | 25100.0 (0.777s) | 25176.0 (0.788s) | 

In [15]:
function random_feasible_solution(kp::KnapsackProblem; rng::AbstractRNG=Random.default_rng(), assign_probability::Float64=0.7)
    n = length(kp.weights)
    x = fill(0, n)
    loads = zeros(Int, length(kp.capacities))
    items = randperm(rng, n)

    for i in items
        if rand(rng) > assign_probability
            continue
        end
        bins = randperm(rng, length(kp.capacities))
        for b in bins
            if loads[b] + kp.weights[i] <= kp.capacities[b]
                x[i] = b
                loads[b] += kp.weights[i]
                break
            end
        end
    end

    return x
end

random_feasible_solution (generic function with 1 method)

In [8]:
function solve_tabu_lengths(kp::KnapsackProblem;
    tabu_lengths::Vector{Int}=[1, 2, 5],
    iteration_limit::Int=100000,
    seed::Int=GLOBAL_SEED,
    random_start::Bool=true)

    out = Dict{Int, Any}()
    for (idx, L) in enumerate(tabu_lengths)
        rng = MersenneTwister(seed + 100 * idx)
        x0 = random_start ? random_feasible_solution(kp; rng=rng) : fill(0, length(kp.weights))

        st = TabuState(kp, x0; buffer_length=L)
        t0 = time()
        sol, iter_num = solve_tabu(kp, st; iteration_limit=iteration_limit)
        elapsed = time() - t0

        out[L] = (
            assignment = sol,
            profit = -st.best_seen_obj,
            elapsed_seconds = elapsed,
            iterations = iter_num,
        )
    end
    return out
end


solve_tabu_lengths (generic function with 1 method)

## Porównanie metod - eksperymenty:

In [26]:
function compare_methods_on_files(files::Vector{String};
    model_path::String="knapsack_report.mzn",
    solver::String="coinbc",
    tabu_lengths::Vector{Int}=[1, 2, 5],
    iteration_limit::Int=100000)

    rows = NamedTuple[]

    for path in files
        kp = load_knapsack_from_dzn(path)

        mz = run_minizinc_from_julia(model_path, path; solver=solver)
        tabu = solve_tabu_lengths(kp; tabu_lengths=tabu_lengths, iteration_limit=iteration_limit)

        println("\n=== $(basename(path)) ===")
        n_items = length(kp.weights)
        n_knapsacks = length(kp.capacities)
        println("Statystyki instancji: Przedmiotów: $n_items, Plecaków: $n_knapsacks")
        println("Pojemności plecaków: $(kp.capacities)")

        if !isnothing(mz.profit)
            num_selected = count(x -> x > 0, mz.assignment)
            
            println("\nSzczegóły rozwiązania MiniZinc:")
            println("Liczba wybranych przedmiotów: $num_selected")
            
            for j in 1:n_knapsacks
                idx = findall(x -> x == j, mz.assignment)
                
                val_in_bag = sum(kp.profits[idx], init=0)
                weight_in_bag = sum(kp.weights[idx], init=0)
                
                println(@sprintf("  Plecak %d: Wartość = %d, Waga = %d/%d, Przedmioty: %d", 
                        j, val_in_bag, weight_in_bag, kp.capacities[j], length(idx)))
            end
        end

        println(@sprintf("MiniZinc: profit=%s, time=%.3fs, rules=%s", string(mz.profit), mz.elapsed_seconds, string(mz.packing_rules_ok)))
        for L in tabu_lengths
            t = tabu[L]
            println(@sprintf("Tabu L=%d: profit=%d, time=%.3fs, iterations=%d", L, t.profit, t.elapsed_seconds, t.iterations))
            push!(rows, (
                file = basename(path),
                method = "Tabu_L$(L)",
                profit = t.profit,
                elapsed_seconds = t.elapsed_seconds
            ))
        end

        push!(rows, (
            file = basename(path),
            method = "MiniZinc",
            profit = mz.profit === nothing ? -1 : mz.profit,
            elapsed_seconds = mz.elapsed_seconds
        ))
    end

    return rows
end

compare_methods_on_files (generic function with 1 method)

In [27]:
if !@isdefined(generated_files)
    generated_files = sort(filter(f -> endswith(f, ".dzn"), readdir("problems"; join=true)))
end

comparison_rows = compare_methods_on_files(generated_files;
    model_path="knapsack_report.mzn",
    solver="coinbc",
    tabu_lengths=[1, 2, 5],
    iteration_limit=100000
)

println("\nLiczba rekordów porównania: ", length(comparison_rows))
first_n = min(length(comparison_rows), 8)
println("Przykładowe rekordy:")
for i in 1:first_n
    println(comparison_rows[i])
end


=== kp_01.dzn ===
Statystyki instancji: Przedmiotów: 35, Plecaków: 3
Pojemności plecaków: [211, 217, 187]

Szczegóły rozwiązania MiniZinc:
Liczba wybranych przedmiotów: 16
  Plecak 1: Wartość = 1308, Waga = 211/211, Przedmioty: 4
  Plecak 2: Wartość = 1661, Waga = 217/217, Przedmioty: 8
  Plecak 3: Wartość = 1138, Waga = 186/187, Przedmioty: 4
MiniZinc: profit=4107, time=0.322s, rules=true
Tabu L=1: profit=3167, time=0.088s, iterations=6
Tabu L=2: profit=3317, time=0.078s, iterations=6
Tabu L=5: profit=3297, time=0.085s, iterations=16

=== kp_02.dzn ===
Statystyki instancji: Przedmiotów: 45, Plecaków: 3
Pojemności plecaków: [234, 232, 225]

Szczegóły rozwiązania MiniZinc:
Liczba wybranych przedmiotów: 20
  Plecak 1: Wartość = 1593, Waga = 234/234, Przedmioty: 7
  Plecak 2: Wartość = 1869, Waga = 232/232, Przedmioty: 6
  Plecak 3: Wartość = 1334, Waga = 225/225, Przedmioty: 7
MiniZinc: profit=4796, time=1.024s, rules=true
Tabu L=1: profit=3554, time=0.130s, iterations=3
Tabu L=2: profi

W każdym z przypadków MiniZinc znalazł najlepsze rozwiązanie

| Plik | Tabu_L1 | Tabu_L2 | Tabu_L5 | MiniZinc |
| :--- | ------- | ------- | ------- | -------- |
| kp_01.dzn | 3167 | 3317 | 3297 | 4107 |
| kp_02.dzn | 3554 | 3542 | 3727 | 4796 |
| kp_03.dzn | 5310 | 5237 | 5393 | 6915 |
| kp_04.dzn | 7191 | 7274 | 6760 | 9291 |
| kp_05.dzn | 8519 | 8612 | 8946 | 10866 |
| kp_06.dzn | 10101 | 10338 | 9827 | 12220 |
| kp_07.dzn | 10635 | 11685 | 12030 | 14865 |
| kp_08.dzn | 12681 | 13447 | 12645 | 17163 |
| kp_09.dzn | 16903 | 15084 | 15516 | 21169 |
| kp_10.dzn | 19846 | 20001 | 19228 | 26784 |

# Na 4.0

In [9]:
function possible_moves2(p::KnapsackProblem, x::Vector{Int})
    move_list = Tuple{Int,Int}[]
    current_weights = zeros(Int, length(p.capacities))
    # add item
    for (item_id, bin_id) in enumerate(x)
        if bin_id > 0 # 0 oznacza brak przypisania
            current_weights[bin_id] += p.weights[item_id]
        end
    end    

    for i in eachindex(x)
        if x[i] > 0
            # możliwość usunięcia i z plecaka x[i]
            push!(move_list, (i, 0))
        end
        for k in eachindex(p.capacities)
            # przeniesienie do plecaka
            if x[i] == 0 && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i,k))
            end
            # dodatkowa możliwość - przeniesienie do innego plecaka
            if x[i] != k && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i, k))
            end
        end
    end

    return move_list
end

possible_moves2 (generic function with 1 method)

In [ ]:
function solve_tabu2(p, s::TabuState; iteration_limit::Int=100)
    iter_best_move = 0
    while s.iter < iteration_limit
        moves = possible_moves2(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                continue
            end
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        if best_move == 0
            break
        end
        chosen_move = moves[best_move]
        item_id = chosen_move[1]
        prev_bin = s.current[item_id]
        apply!(s.current, chosen_move)
        push!(s.tabu_buffer, invert_move(p, moves[best_move], prev_bin))
        if best_move_obj < s.best_seen_obj
            iter_best_move = s.iter
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen, iter_best_move
end

solve_tabu2 (generic function with 1 method)

In [11]:
using Statistics
using Printf

function run_comprehensive_comparison(param_files, iteration_limit=10000)
    tabu_lengths = [1, 2, 5]
    results_profit = Dict()
    results_time = Dict()

    for file in param_files
        fname = basename(file)
        kp = load_knapsack_from_dzn(file)
        
        println("Przetwarzanie: $fname...")

        mz = run_minizinc_from_julia("knapsack_report.mzn", file)
        results_profit[(fname, "MiniZinc")] = [mz.profit === nothing ? 0 : mz.profit]
        results_time[(fname, "MiniZinc")] = [mz.elapsed_seconds]

        for L in tabu_lengths
            method_name = "Tabu_L$L"
            profits = Float64[]
            times = Float64[]

            for _ in 1:10
                stats = @timed begin
                    x0 = fill(0, length(kp.weights))
                    st = TabuState(kp, x0; buffer_length = L)
                    sol, iter_num = solve_tabu2(kp, st; iteration_limit = iteration_limit)
                    -st.best_seen_obj
                end
                push!(profits, stats.value)
                push!(times, stats.time)
            end
            
            results_profit[(fname, method_name)] = profits
            results_time[(fname, method_name)] = times
        end
    end

    
    methods = ["MiniZinc", "Tabu_L1", "Tabu_L2", "Tabu_L5"]
    files = [basename(f) for f in param_files]

    println("\n### Podsumowanie: Średni Profit | Średni Czas [s]")
    
    # Nagłówek
    header = "| Zestaw parametrów | " * join(methods, " | ") * " |"
    sep = "| :--- | " * join([repeat("-", length(m)) for m in methods], " | ") * " |"
    println(header)
    println(sep)

    for f in files
        row = "| $f "
        for m in methods
            avg_p = mean(get(results_profit, (f, m), [0]))
            avg_t = mean(get(results_time, (f, m), [0]))
            # Formatujemy: Profit (Czas)
            row *= @sprintf("| %.1f (%.3fs) ", avg_p, avg_t)
        end
        println(row * "|")
    end
end

run_comprehensive_comparison (generic function with 2 methods)

In [38]:
run_comprehensive_comparison(generated_files)

Przetwarzanie: kp_01.dzn...
Przetwarzanie: kp_02.dzn...
Przetwarzanie: kp_03.dzn...
Przetwarzanie: kp_04.dzn...
Przetwarzanie: kp_05.dzn...
Przetwarzanie: kp_06.dzn...
Przetwarzanie: kp_07.dzn...
Przetwarzanie: kp_08.dzn...
Przetwarzanie: kp_09.dzn...
Przetwarzanie: kp_10.dzn...

### Podsumowanie: Średni Profit | Średni Czas [s]
| Zestaw parametrów | MiniZinc | Tabu_L1 | Tabu_L2 | Tabu_L5 |
| :--- | -------- | ------- | ------- | ------- |
| kp_01.dzn | 4107.0 (0.828s) | 4017.0 (0.025s) | 4017.0 (0.007s) | 4017.0 (0.016s) |
| kp_02.dzn | 4796.0 (1.060s) | 4560.0 (0.009s) | 4545.0 (0.010s) | 4555.0 (0.012s) |
| kp_03.dzn | 6915.0 (0.607s) | 6707.0 (0.012s) | 6714.0 (0.012s) | 6722.0 (0.015s) |
| kp_04.dzn | 9291.0 (1.091s) | 8832.0 (0.022s) | 8863.0 (0.022s) | 9010.0 (0.022s) |
| kp_05.dzn | 10866.0 (3.595s) | 10401.0 (0.023s) | 10401.0 (0.022s) | 10401.0 (0.032s) |
| kp_06.dzn | 12220.0 (49.438s) | 11818.0 (0.074s) | 11818.0 (0.077s) | 11839.0 (0.068s) |
| kp_07.dzn | 14865.0 (1.356s) 

### Podsumowanie: Średni Profit | Średni Czas [s] (minizinc + druga wersja tabu search)
| Zestaw parametrów | MiniZinc | Tabu_L1 | Tabu_L2 | Tabu_L5 |
| :--- | -------- | ------- | ------- | ------- |
| kp_01.dzn | 4107.0 (0.828s) | 4017.0 (0.025s) | 4017.0 (0.007s) | 4017.0 (0.016s) |
| kp_02.dzn | 4796.0 (1.060s) | 4560.0 (0.009s) | 4545.0 (0.010s) | 4555.0 (0.012s) |
| kp_03.dzn | 6915.0 (0.607s) | 6707.0 (0.012s) | 6714.0 (0.012s) | 6722.0 (0.015s) |
| kp_04.dzn | 9291.0 (1.091s) | 8832.0 (0.022s) | 8863.0 (0.022s) | 9010.0 (0.022s) |
| kp_05.dzn | 10866.0 (3.595s) | 10401.0 (0.023s) | 10401.0 (0.022s) | 10401.0 (0.032s) |
| kp_06.dzn | 12220.0 (49.438s) | 11818.0 (0.074s) | 11818.0 (0.077s) | 11839.0 (0.068s) |
| kp_07.dzn | 14865.0 (1.356s) | 13959.0 (0.089s) | 13959.0 (0.085s) | 14090.0 (0.104s) |
| kp_08.dzn | 17163.0 (3.176s) | 15809.0 (0.144s) | 15862.0 (0.099s) | 15954.0 (0.153s) |
| kp_09.dzn | 21169.0 (51.541s) | 19692.0 (0.229s) | 19715.0 (0.243s) | 19787.0 (0.317s) |
| kp_10.dzn | 26784.0 (44.999s) | 25100.0 (0.477s) | 25100.0 (0.444s) | 25176.0 (0.485s) |

W porównaniu z tabu searchem bez przenoszenia przedmiotów z plecaka do pleacak, nowa wersja ma prawie że identyczne wyniki funkcji celu, natomiast wywołuje się szybciej.

## Na 5.0

## Symulowane wyzazanie - TabuSearch

In [ ]:
mutable struct SAState{P, TF}
    current::P
    best_seen::P
    best_seen_obj::TF
    temp::Float64
    iter::Int
end
function SAState(p, x0; initial_temp::Float64=100.0)
    obj = objective(p, x0)
    return SAState{typeof(x0), typeof(obj)}(
        copy(x0),
        copy(x0),
        obj,
        initial_temp,
        1
    )
end

function solve_simulated_annealing(p, s::SAState;
                                   max_iter=10000,
                                   cooling_rate=0.995,
                                   rng::AbstractRNG=Random.default_rng())
    iter_best_move = 0
    while s.iter < max_iter

        moves = possible_moves(p, s.current)
        if isempty(moves) break end
        move = rand(rng, moves)
        old_obj = objective(p, s.current)

        test_state = copy(s.current)
        apply!(test_state, move)
        new_obj = objective(p, test_state)

        ΔE = new_obj - old_obj

        if ΔE < 0 || rand(rng) < exp(-ΔE / s.temp)
            s.current = test_state

            if new_obj < s.best_seen_obj
                iter_best_move = s.iter
                s.best_seen = copy(s.current)
                s.best_seen_obj = new_obj
            end
        end

        s.temp *= cooling_rate
        s.iter += 1
    end
    return s.best_seen, iter_best_move
end

solve_simulated_annealing (generic function with 1 method)

In [69]:

function test2(kp)
    x0 = fill(0, length(kp.weights))
    st = SAState(kp, x0)
    sol = solve_simulated_annealing(kp, st; max_iter=1000000)

    println("Full solution: ", sol)
    
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end
@time begin
test2(kp1)
end

Iteracja w której zostało osiągnięte najlepsze rozwiązanie: 484
Full solution: [0, 0, 0, 3, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 1, 0, 0, 0, 2, 2, 1, 0, 2, 1, 2, 0, 1, 0, 1, 3, 3, 0, 0, 0, 0, 3, 0, 0, 0, 3, 0, 3, 1, 0, 0, 0, 2, 0, 3, 0, 2, 0, 1, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 3, 1, 1, 0, 0, 2, 0, 3]
Best objective: -2057
Last iteration: 1000000
  1.462873 seconds (10.10 M allocations: 4.163 GiB, 13.64% gc time, 3.52% compilation time)


## Testy

In [13]:
function assignment_stats(kp::KnapsackProblem, assignment::Vector{Int})
    k = length(kp.capacities)
    counts = zeros(Int, k)
    profits = zeros(Int, k)
    loads = zeros(Int, k)

    for i in eachindex(assignment)
        b = assignment[i]
        if b > 0
            counts[b] += 1
            profits[b] += kp.profits[i]
            loads[b] += kp.weights[i]
        end
    end

    selected = sum(counts)
    outside = count(==(0), assignment)
    return (counts=counts, profits=profits, loads=loads, selected=selected, outside=outside)
end

function random_feasible_solution(kp::KnapsackProblem; rng::AbstractRNG=Random.default_rng(), assign_probability::Float64=0.7)
    n = length(kp.weights)
    x = fill(0, n)
    loads = zeros(Int, length(kp.capacities))
    items = randperm(rng, n)

    for i in items
        if rand(rng) > assign_probability
            continue
        end
        bins = randperm(rng, length(kp.capacities))
        for b in bins
            if loads[b] + kp.weights[i] <= kp.capacities[b]
                x[i] = b
                loads[b] += kp.weights[i]
                break
            end
        end
    end

    return x
end


function stage5_compare_sa(files::Vector{String};
    model_path::String="knapsack_report.mzn",
    solver::String="coinbc",
    iteration_limit::Int=100000,
    sa_max_iter::Int=100000,
    cooling_rates::Vector{Float64}=[0.990, 0.995, 0.998],
    seed::Int=GLOBAL_SEED)

    rows = NamedTuple[]
    for (i, path) in enumerate(sort(files))
        kp = load_knapsack_from_dzn(path)
        mz = run_minizinc_from_julia(model_path, path; solver=solver)
        tabu = solve_tabu_lengths(kp;
            tabu_lengths=[1,2,5],
            iteration_limit=iteration_limit,
            seed=seed + 2000 * i,
            random_start=true)

        for (j, cr) in enumerate(cooling_rates)
            rng = MersenneTwister(seed + 2000 * i + 100 * j)
            x0 = random_feasible_solution(kp; rng=rng)
            st = SAState(kp, x0)
            t0 = time()
            sol, iter_best_move = solve_simulated_annealing(kp, st;
                max_iter=sa_max_iter,
                cooling_rate=cr,
                rng=rng)
            _ = sol
            sa_elapsed = time() - t0

            push!(rows, (
                file = basename(path),
                method = "SA_$(cr)",
                profit = -st.best_seen_obj,
                elapsed_seconds = sa_elapsed,
                minizinc_profit = mz.profit,
                tabu_l1_profit = tabu[1].profit,
                tabu_l2_profit = tabu[2].profit,
                tabu_l5_profit = tabu[5].profit
            ))
        end
    end
    return rows
end

stage5_compare_sa (generic function with 1 method)

In [14]:
set_seed!()

if !@isdefined(generated_files)
    generated_files = sort(filter(f -> endswith(f, ".dzn"), readdir("problems"; join=true)))
end


stage5_rows = stage5_compare_sa(generated_files; seed=GLOBAL_SEED, sa_max_iter=100000)
println("Stage 5 - liczba rekordów: ", length(stage5_rows))
println(stage5_rows[1])

Stage 5 - liczba rekordów: 30
(file = "kp_01.dzn", method = "SA_0.99", profit = 3153, elapsed_seconds = 0.026000022888183594, minizinc_profit = 4107, tabu_l1_profit = 3130, tabu_l2_profit = 3109, tabu_l5_profit = 3248)


### Porównanie Simulated Annealing z MiniZinc i Tabu Search
| Plik | MiniZinc | Tabu (L5) | SA_0.99 | SA_0.995 | SA_0.998 |
| :--- | :--- | :--- | ------- | -------- | -------- |
| **kp_01.dzn** | 4107 | 3248 | 3153 (0.026s) | 3366 (0.033s) | 3343 (0.027s) |
| **kp_02.dzn** | 4796 | 3303 | 3551 (0.036s) | 4142 (0.031s) | 3633 (0.033s) |
| **kp_03.dzn** | 6915 | 5366 | 5819 (0.101s) | 5267 (0.098s) | 5493 (0.092s) |
| **kp_04.dzn** | 9291 | 7650 | 7114 (0.084s) | 7410 (0.099s) | 7752 (0.101s) |
| **kp_05.dzn** | 10866 | 8604 | 8901 (0.086s) | 9453 (0.080s) | 9044 (0.086s) |
| **kp_06.dzn** | 12220 | 10125 | 8730 (0.094s) | 10165 (0.094s) | 10781 (0.098s) |
| **kp_07.dzn** | 14865 | 11645 | 12171 (0.117s) | 12392 (0.100s) | 12542 (0.104s) |
| **kp_08.dzn** | 17163 | 13305 | 12274 (0.106s) | 13086 (0.100s) | 14168 (0.117s) |
| **kp_09.dzn** | 21169 | 16068 | 17232 (0.117s) | 16275 (0.126s) | 16345 (0.224s) |
| **kp_10.dzn** | 26784 | 20988 | 20025 (0.145s) | 21024 (0.136s) | 21698 (0.143s) |